In [3]:
from bs4 import BeautifulSoup, Tag, NavigableString
from markdownify import markdownify
import glob, re, sqlite3, os

In [8]:
SECTION_MAP = {
    '독립된감사인의감사보고서': '감사보고서',
    '첨부재무제표': '재무제표',
    '재무제표': '재무제표',
    '주석': '주석',
    '내부회계관리제도': '내부회계관리',
    '외부감사실시내용': '외부감사',
}

FS_KEYWORDS = [
    '재무상태표',
    '손익계산서',
    '포괄손익계산서',
    '자본변동표',
    '현금흐름표'
]

DEFAULT_COMPANY = "삼성전자주식회사"


# =========================
# 공통 유틸
# =========================
def clean_text(text):
    text = re.sub(r'["“”‘’]', '', text or '')
    text = text.replace('\xa0', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_compact(text):
    return re.sub(r'\s+', '', text or '').strip()

def get_year_from_filename(filepath):
    m = re.search(r'(\d{4})', os.path.basename(filepath))
    return int(m.group(1)) if m else None

def get_doc_id(filepath):
    year = get_year_from_filename(filepath)
    return f"samsung_{year}" if year else None

def normalize_company_name(text):
    t = clean_compact(text)
    if "삼성전자주식회사" in t or "삼성전자주식회사" in t.replace(" ", ""):
        return "삼성전자주식회사"
    if "삼성전자주식회사" in text or "삼성전자 주식회사" in text:
        return "삼성전자주식회사"
    return DEFAULT_COMPANY

def extract_company(soup):
    body_text = clean_text(soup.get_text("\n", strip=True))
    return normalize_company_name(body_text)


# =========================
# section context
# =========================
def normalize_section_title(raw_title):
    t = clean_compact(raw_title).replace("(첨부)", "")
    for key, val in SECTION_MAP.items():
        if key in t:
            return val
    return "기타"

def get_section_context(table):
    """
    표 기준으로 가장 가까운 이전 h1/h2/h3/h4 id='toc_*' 헤더를 찾는다.
    반환:
    {
        section_id,
        section_title_raw,
        section_title_std
    }
    """
    prev_h = table.find_previous(['h1', 'h2', 'h3', 'h4'])
    while prev_h:
        hid = prev_h.get("id", "")
        if hid.startswith("toc_"):
            raw = clean_text(prev_h.get_text(" ", strip=True))
            return {
                "section_id": hid,
                "section_title_raw": raw,
                "section_title_std": normalize_section_title(raw)
            }
        prev_h = prev_h.find_previous(['h1', 'h2', 'h3', 'h4'])

    return {
        "section_id": None,
        "section_title_raw": None,
        "section_title_std": "기타"
    }

def get_section(table):
    return get_section_context(table)["section_title_std"]


# =========================
# 제목 판별용 함수
# =========================
def extract_p_lines(p_tag):
    # span이 쪼개져 있는 경우도 한 줄로 먼저 합침
    raw = p_tag.get_text(" ", strip=True)
    raw = raw.replace("= $0", " ")   # 개발자도구에서 보이는 선택 표시 제거 대비
    raw = clean_text(raw)

    # "4 . 현금및현금성자산" -> "4. 현금및현금성자산"
    raw = re.sub(r'(\d+)\s*\.\s*', r'\1. ', raw)

    return [raw] if raw else []

def is_level1_title(text):
    # 1. 일반적 사항 / 18. 자본금
    # 2.14 금융부채 같은 것은 제외
    text = clean_text(text)
    text = re.sub(r'(\d+)\s*\.\s*', r'\1. ', text)
    text = re.split(r'\s*[:：]\s*', text, maxsplit=1)[0].strip()
    return bool(re.match(r'^\d+\.\s*[^\d\s].*', text))

def is_level2_title(text):
    text = clean_text(text)
    return bool(re.match(r'^[가-하]\.\s*.+', text))

def is_level3_title(text):
    text = clean_text(text)
    return bool(re.match(r'^\(\d+\)\s*.+', text))

def is_level4_title(text):
    text = clean_text(text)
    return bool(re.match(r'^\d+\)\s*.+', text))


# =========================
# 주석 title map 구축
# =========================
def build_note_title_map(soup):
    """
    주석 섹션에서 제목/세부제목을 순서대로 스캔하여
    각 P 태그에 대해 현재 문맥(subsection_path)을 기억한다.
    """

    note_title_map = {}
    current_l1 = None   # 예: 18. 자본금
    current_l2 = None   # 예: 가. 세부항목
    current_l3 = None   # 예: (1) 세부항목
    order = 0

    note_section = None
    for h in soup.find_all(['h1', 'h2', 'h3', 'h4']):
        hid = h.get("id", "")
        if hid.startswith("toc_"):
            title = clean_compact(h.get_text(" ", strip=True))
            if "주석" in title:
                note_section = h
                break

    if note_section is None:
        return note_title_map

    node = note_section.find_next()
    while node:
        if isinstance(node, Tag) and node.name in ['h1', 'h2', 'h3', 'h4']:
            hid = node.get("id", "")
            if hid.startswith("toc_") and node != note_section:
                break

        if isinstance(node, Tag) and node.name == "p":
            lines = extract_p_lines(node)
            for line in lines:
                txt = clean_text(line)
                txt = re.sub(r'(\d+)\s*\.\s*', r'\1. ', txt)
                txt = re.split(r'\s*[:：]\s*', txt, maxsplit=1)[0].strip()

                if is_level1_title(txt):
                    current_l1 = txt
                    current_l2 = None
                    current_l3 = None
                    order += 1

                elif is_level2_title(txt):
                    current_l2 = txt
                    current_l3 = None
                    order += 1

                elif is_level3_title(txt):
                    current_l3 = txt
                    order += 1

                path_parts = [x for x in [current_l1, current_l2, current_l3] if x]
                subsection_path = " > ".join(path_parts) if path_parts else None

                note_no = None
                note_title = None
                if current_l1:
                    m = re.match(r'^(\d+)\.\s*(.+)$', current_l1)
                    if m:
                        note_no = m.group(1)
                        note_title = clean_text(m.group(2))

                note_title_map[id(node)] = {
                    "note_no": note_no,
                    "note_title": note_title,
                    "subsection_path": subsection_path,
                    "context_order": order
                }

        node = node.find_next()

    return note_title_map


# =========================
# 표 근처 문맥 찾기
# =========================
def get_nearest_note_context(table, note_title_map):
    """
    표 바로 앞쪽 p 태그를 역방향으로 훑으면서 가장 가까운 주석 문맥을 반환
    """
    prev = table.find_previous()
    while prev:
        if isinstance(prev, Tag) and prev.name == "p":
            info = note_title_map.get(id(prev))
            if info:
                return info
        if isinstance(prev, Tag) and prev.name in ['h1', 'h2', 'h3', 'h4']:
            hid = prev.get("id", "")
            if hid.startswith("toc_"):
                break
        prev = prev.find_previous()
    return {
        "note_no": None,
        "note_title": None,
        "subsection_path": None,
        "context_order": None
    }


# =========================
# 내부회계 제목
# =========================
def get_internal_control_title(soup):
    body = clean_text(soup.get_text("\n", strip=True))
    if "독립된 감사인의 내부회계관리제도 감사보고서" in body:
        return "독립된 감사인의 내부회계관리제도 감사보고서"
    if "독립된 감사인의 내부회계관리제도 검토보고서" in body:
        return "독립된 감사인의 내부회계관리제도 검토보고서"
    return "내부회계관리제도"


# =========================
# 표 제목 추출
# =========================
def get_fs_title(table):
    prev_nb = table.find_previous_sibling('table', class_='nb')
    if not prev_nb:
        return '제목없음'

    raw = prev_nb.get_text(separator=' ', strip=True)
    raw = re.sub(r'\(단위.*?\)', '', raw)
    raw = clean_text(raw)
    compact = clean_compact(raw)

    for kw in FS_KEYWORDS:
        if kw in compact:
            return kw

    return '제목없음'


def get_table_title(table, section, note_title_map=None, internal_title=None, soup=None):
    """
    표 제목 우선순위
    1) 재무제표 표면 FS title
    2) 내부회계/외부감사 표면 section용 제목
    3) 주석 섹션이면 가까운 주석 문맥(subsection_path)
    4) 직전 p 텍스트
    """
    if section == '재무제표':
        fs_title = get_fs_title(table)
        if fs_title != '제목없음':
            return fs_title

    if section == '내부회계관리':
        return internal_title or '내부회계관리제도'

    if section == '외부감사':
        return '외부감사실시내용'

    if section == '주석' and note_title_map is not None:
        ctx = get_nearest_note_context(table, note_title_map)
        if ctx["subsection_path"]:
            return ctx["subsection_path"]
        if ctx["note_title"]:
            return ctx["note_title"]

    prev_p = table.find_previous_sibling('p')
    if prev_p:
        txt = clean_text(prev_p.get_text(" ", strip=True))
        if txt:
            return txt[:200]

    return '제목없음'


# =========================
# 각주 / markdown
# =========================
def get_footnotes(table):
    footnotes = []

    nxt = table.find_next_sibling()
    while nxt:
        if isinstance(nxt, Tag):
            if nxt.name == 'table' and 'TABLE' in (nxt.get('class') or []):
                break
            if nxt.name in ['h1', 'h2', 'h3', 'h4']:
                break

            txt = clean_text(nxt.get_text(" ", strip=True))
            if txt and re.match(r'^\(\*(?:\d+)?\)', txt):
                footnotes.append(txt)

        nxt = nxt.find_next_sibling()

    return footnotes


def table_to_markdown(table):
    md = markdownify(str(table), heading_style='ATX')
    md = re.sub(r'\n{3,}', '\n\n', md).strip()

    footnotes = get_footnotes(table)
    if footnotes:
        md += '\n\n' + '\n'.join(footnotes)

    return md

def table_to_markdown_with_row_headers(table):
    md_table = table_to_markdown(table)

    row_headers = extract_row_headers(table)
    row_headers_md = row_headers_to_markdown(row_headers)

    return md_table, row_headers_md

# =========================
# 표 행 제목 추출
# =========================
def extract_row_headers(table):
    """테이블에서 행 제목(첫 번째 컬럼)만 추출"""
    headers = []

    for tr in table.find_all("tr"):
        cells = tr.find_all(["td", "th"])
        if not cells:
            continue

        first_cell = cells[0].get_text(strip=True)

        # 빈 값 제외
        if first_cell:
            headers.append(first_cell)

    return headers

def row_headers_to_markdown(headers):
    """행 제목 리스트 → 1열 Markdown 표"""
    if not headers:
        return ""

    lines = ["| row_header |", "|---|"]
    lines += [f"| {h} |" for h in headers]
    return "\n".join(lines)

# =========================
# 표 열 제목 추출
# =========================
def is_data_table(table):
    rows = table.find_all("tr")
    if len(rows) < 2:
        return False

    has_number = False
    for tr in rows[:10]:
        txt = clean_text(tr.get_text(" ", strip=True))
        if re.search(r'[\d,()\-]+', txt):
            has_number = True
            break
    return has_number

import re

def is_numeric_like(text):
    """금액/숫자/괄호음수 형태인지 판단"""
    if not text:
        return False
    text = text.strip()
    return bool(re.fullmatch(r'[\d,\.\-\(\)]+', text))

def clean_cell_text(cell):
    return clean_text(cell.get_text(" ", strip=True))

def expand_cells(cells):
    expanded = []
    for cell in cells:
        text = clean_cell_text(cell)
        colspan = int(cell.get("colspan", 1))
        expanded.extend([text] * colspan)
    return expanded

def get_header_rows(table, max_check_rows=5):
    """
    실제 헤더행만 수집:
    - 숫자 비중이 높은 행이 나오면 데이터행으로 보고 중단
    - 최대 앞 5행까지만 검사
    """
    trs = table.find_all("tr")
    header_rows = []

    for tr in trs[:max_check_rows]:
        cells = tr.find_all(["th", "td"])
        if not cells:
            continue

        row = expand_cells(cells)
        nonempty = [x for x in row if x]

        if not nonempty:
            continue

        numeric_cnt = sum(is_numeric_like(x) for x in nonempty)
        numeric_ratio = numeric_cnt / len(nonempty)

        # 숫자 비중이 높으면 데이터행으로 간주
        if numeric_ratio >= 0.5:
            break

        header_rows.append(row)

    return header_rows

def merge_header_rows(header_rows):
    """
    다중 헤더를 세로로 합침.
    같은 값 반복은 제거.
    """
    if not header_rows:
        return []

    max_len = max(len(r) for r in header_rows)
    normalized = []

    for row in header_rows:
        if len(row) < max_len:
            row = row + [""] * (max_len - len(row))
        normalized.append(row)

    merged = []
    for col_idx in range(max_len):
        parts = []
        for row_idx in range(len(normalized)):
            val = normalized[row_idx][col_idx].strip()
            if not val:
                continue
            if not parts or parts[-1] != val:
                parts.append(val)

        merged.append("_".join(parts) if parts else "")

    return merged

def extract_column_headers(table):
    header_rows = get_header_rows(table)

    if not header_rows:
        return []

    headers = merge_header_rows(header_rows)

    # 완전 빈 헤더 제거는 하지 말고 원형 유지
    return headers

def column_headers_to_markdown(headers):
    if not headers:
        return ""

    lines = ["| column_header |", "|---|"]
    lines += [f"| {h} |" for h in headers]
    return "\n".join(lines)

# =========================
# 최종 row builder
# =========================
def build_table_record(tbl, filepath, idx, soup, note_title_map, internal_title):
    year = get_year_from_filename(filepath)
    doc_id = get_doc_id(filepath)
    company = extract_company(soup)

    sec = get_section_context(tbl)
    note_ctx = get_nearest_note_context(tbl, note_title_map)
    
    title = get_table_title(
        tbl,
        sec["section_title_std"],
        note_title_map=note_title_map,
        internal_title=internal_title,
        soup=soup
    )

    footnotes = get_footnotes(tbl)
    md, row_headers_md = table_to_markdown_with_row_headers(tbl)
    column_headers_md = None
    if is_data_table(tbl):
        column_headers = extract_column_headers(tbl)
        column_headers_md = column_headers_to_markdown(column_headers)
    html_raw = str(tbl)

    table_id = f"{doc_id}_table_{idx:04d}"

    return (
        table_id,
        doc_id,
        company,
        year,
        os.path.basename(filepath),
        sec["section_id"],
        sec["section_title_std"],
        sec["section_title_raw"],
        note_ctx["note_no"],
        note_ctx["note_title"],
        note_ctx["subsection_path"],
        idx,
        title,
        "\n".join(footnotes) if footnotes else None,
        md,
        html_raw,
        row_headers_md,
        column_headers_md
    )

In [9]:
# ── 전체 파일 파싱 → SQLite 저장 ──
DB_PATH = 'audit_tables_v4.db'

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

cur.execute('DROP TABLE IF EXISTS tables')
cur.execute('''
    CREATE TABLE tables (
        table_id            TEXT PRIMARY KEY,
        doc_id              TEXT,
        company             TEXT,
        year                INTEGER,
        source_file         TEXT,

        section_id          TEXT,
        section_title_std   TEXT,
        section_title_raw   TEXT,

        note_no             TEXT,
        note_title          TEXT,
        subsection_path     TEXT,

        table_order         INTEGER,
        title               TEXT,
        footnotes           TEXT,
        markdown            TEXT,
        html_raw            TEXT,
        row_headers_md      TEXT,
        column_headers_md   TEXT
    )
''')
conn.commit()

htm_files = sorted(glob.glob('*.htm') + glob.glob('*.html'))
print(f'파일 수: {len(htm_files)}')

total_count = 0

for filepath in htm_files:
    with open(filepath, 'rb') as f:
        soup = BeautifulSoup(f.read(), 'html.parser')

    tables = soup.find_all('table', class_='TABLE')
    note_title_map = build_note_title_map(soup)
    internal_title = get_internal_control_title(soup)

    rows = []
    for idx, tbl in enumerate(tables, start=1):
        row = build_table_record(
            tbl=tbl,
            filepath=filepath,
            idx=idx,
            soup=soup,
            note_title_map=note_title_map,
            internal_title=internal_title
        )
        rows.append(row)

    cur.executemany('''
        INSERT INTO tables VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', rows)

    conn.commit()
    total_count += len(rows)
    print(f'{filepath} → 표 {len(rows)}개 저장')

conn.close()
print(f'\n총 {total_count}개 표 저장 완료 → {DB_PATH}')

파일 수: 11
감사보고서_2014.htm → 표 122개 저장
감사보고서_2015.htm → 표 124개 저장
감사보고서_2016.htm → 표 122개 저장
감사보고서_2017.htm → 표 119개 저장
감사보고서_2018.htm → 표 127개 저장
감사보고서_2019.htm → 표 120개 저장
감사보고서_2020.htm → 표 116개 저장
감사보고서_2021.htm → 표 114개 저장
감사보고서_2022.htm → 표 114개 저장
감사보고서_2023.htm → 표 114개 저장
감사보고서_2024.htm → 표 116개 저장

총 1308개 표 저장 완료 → audit_tables_v4.db


In [4]:
# ── 저장 결과 확인 ──
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

cur.execute('SELECT COUNT(*) FROM tables')
print(f'총 저장된 표: {cur.fetchone()[0]}개')

print('\n[연도별 표 개수]')
cur.execute('SELECT year, COUNT(*) FROM tables GROUP BY year ORDER BY year')
for row in cur.fetchall():
    print(f'  {row[0]}년: {row[1]}개')

print('\n[섹션별 표 개수]')
cur.execute('SELECT section_title_std, COUNT(*) FROM tables GROUP BY section_title_std ORDER BY COUNT(*) DESC')
for row in cur.fetchall():
    print(f'  {row[0]}: {row[1]}개')

print('\n[샘플 - 2020년 주석 표 5개]')
cur.execute('''
    SELECT table_id, year, section_title_std, note_no, note_title, subsection_path, title
    FROM tables
    WHERE year = 2020
    ORDER BY table_order
    LIMIT 5
''')
for row in cur.fetchall():
    print(row)

conn.close()

총 저장된 표: 1308개

[연도별 표 개수]
  2014년: 122개
  2015년: 124개
  2016년: 122개
  2017년: 119개
  2018년: 127개
  2019년: 120개
  2020년: 116개
  2021년: 114개
  2022년: 114개
  2023년: 114개
  2024년: 116개

[섹션별 표 개수]
  주석: 1196개
  재무제표: 55개
  외부감사: 40개
  감사보고서: 11개
  내부회계관리: 6개

[샘플 - 2020년 주석 표 5개]
('samsung_2020_table_0001', 2020, '감사보고서', None, None, None, '제목없음')
('samsung_2020_table_0002', 2020, '재무제표', None, None, None, '재무상태표')
('samsung_2020_table_0003', 2020, '재무제표', None, None, None, '손익계산서')
('samsung_2020_table_0004', 2020, '재무제표', None, None, None, '손익계산서')
('samsung_2020_table_0005', 2020, '재무제표', None, None, None, '자본변동표')
